In [8]:
import pandas as pd
import numpy as np
import optuna
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm
from xgboost import XGBRegressor

In [2]:
apple = pd.read_parquet('Apple_features.parquet')
btc = pd.read_parquet('BTC_features.parquet')
qqq = pd.read_parquet('QQQ_features.parquet')
spx = pd.read_parquet('spx_features.parquet')
tesla = pd.read_parquet('Tesla_features.parquet')
dfs = {
    "apple": apple,
    "btc": btc,
    "qqq": qqq,
    "spx": spx,
    "tesla": tesla,}


In [3]:
optuna.logging.disable_default_handler()

In [17]:
targets = ["target_3", "target_5", "target_10", "target_30", "target_60"]

em_map = {
    "target_3": 2,
    "target_5": 4,
    "target_10": 9,
    "target_30": 29,
    "target_60": 59,
}

all_target_cols = [
    "target_3", "target_5", "target_10", "target_30", "target_60"
]


In [16]:
def walk_forward(X, Y, dates, model, tr, v, te, em):
    preds = []
    n = len(Y)
    t = np.arange(n)
    for k in range(0,  n - tr - v - 2*em -  te + 1, te):
        train = t[k : k + tr]
        val = t[k + tr + em : k + tr + em + v]
        test = t[k + tr + 2*em + v : k + tr + 2*em + v + te]
        X_train, Y_train = X[train], Y[train]
        X_val, Y_val = X[val], Y[val]
        X_test = X[test]
        Y_hat = model(
            X_train = X_train,
            Y_train = Y_train,
            X_val = X_val,
            Y_val = Y_val,
            X_test = X_test,
            dates = dates
        )
        Y_fact = Y[test]
        fold = pd.DataFrame({
            'Y_true' : Y_fact,
            'Y_pred' : Y_hat
        }, index = dates[test])
        preds.append(fold)
        
    return pd.concat(preds)

In [15]:
def model(X_train, Y_train, dates, X_val = None, Y_val = None, X_test = None, **kwargs):
    reg = tune_model(
        X_train = X_train, 
        Y_train = Y_train,
        X_val = X_val,
        Y_val = Y_val,
        n_trials = kwargs.get('n_trials', 30)
    )  #awesome sklearn model
    return reg.predict(X_test)
    

In [18]:
def tune_model (X_train,Y_train, X_val, Y_val, n_trials = 30):
    def objective(trial):
        params ={ "n_estimators": trial.suggest_int("n_estimators", 50,300),
            "max_depth": trial.suggest_int("max_depth", 2, 5),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 100.0, log=True),
        }
        reg = XGBRegressor(**params,
            objective="reg:squarederror",
            random_state=5,
            n_jobs=-1,
            tree_method="hist"
        )
        reg.fit(X_train, Y_train)
        pred_val = reg.predict(X_val)
        loss = mean_squared_error(Y_val, pred_val) #can use anorher loss
        return loss
    study = optuna.create_study(direction = 'minimize')
    study.optimize(objective, n_trials = n_trials)
    best_reg = XGBRegressor(**study.best_params,
        objective="reg:squarederror",
        random_state=5,
        n_jobs=-1,
        tree_method="hist"
        
    )
    best_reg.fit(np.vstack([X_train, X_val]), np.concatenate([Y_train, Y_val]))
    return best_reg

In [19]:
total_jobs = len(dfs) * len(targets)

with tqdm(total=total_jobs) as pbar:

    for asset_name, df in dfs.items():

        for target in targets:

            data = df.dropna().copy()

            X = data.drop(columns=targets).values
            Y = data[target].values
            dates = data.index

            preds = walk_forward(
                X=X,
                Y=Y,
                dates=dates,
                model=model,
                tr=1260,
                v=252,
                te=21,
                em=em_map[target]
            )

            horizon = target.split("_")[1]

            preds.to_parquet(
                f"{asset_name}_xgb_regression_{horizon}.parquet"
            )

            pbar.set_description(
                f"{asset_name} | {target}"
            )

            pbar.update(1)

  0%|          | 0/25 [00:00<?, ?it/s]